# 📁 Day 3 · 02 · KùzuDB로 배우는 Graph RAG

## 이 실습에서 배우는 것
- KùzuDB 설치와 기본 동작 원리 이해 (서버 없이 동작하는 임베디드 그래프 DB)
- Cypher 쿼리 언어로 노드·관계 만들고 조회하기
- CSV 데이터를 그래프로 대량 적재하기 (COPY FROM)
- LangChain과 결합해서 자연어 질문 → Cypher 자동 생성 → 답변까지 이어지는 Graph RAG 체인 만들기

## 🔍 KùzuDB란

KùzuDB는 **서버를 띄울 필요 없이, 폴더 하나가 곧 데이터베이스가 되는 그래프 DB**입니다.

| 개념 | 설명 |
|------|------|
| Database | `kuzu.Database("./경로")` — 폴더 경로 하나가 DB 전체 |
| Connection | 해당 Database에 쿼리를 보내는 통로 (`kuzu.Connection(db)`) |
| NODE TABLE | 그래프의 점(노드) 종류를 정의하는 스키마 (예: Pest, Crop) |
| REL TABLE | 노드 사이의 관계(엣지)를 정의하는 스키마 (예: Infects) |
| Cypher | 그래프를 조회·생성·수정하는 쿼리 언어 |
| COPY FROM | CSV 파일을 통째로 그래프에 적재하는 명령 |

---
이 노트북은 셀을 위에서 아래로 순서대로 실행하면서 따라가는 구조입니다.

## STEP 1 · 패키지 설치 확인

아래 셀을 실행해서 `kuzu`가 설치되어 있는지 확인합니다.
설치되어 있지 않다면 자동으로 설치를 시도합니다.

In [ ]:
import importlib.util

def ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    if importlib.util.find_spec(pkg) is None:
        import sys
        !{sys.executable} -m pip install -q {pip_name}
    print(f"✅ {pkg} 준비 완료")

ensure("kuzu")
ensure("pandas")


LangChain 연동(STEP 5부터 필요)에 쓸 패키지도 함께 확인합니다.

> ⚠️ **참고**: 독립 패키지였던 `langchain-kuzu`는 archived(유지보수 종료) 상태이고,
> 이를 대신 포함하고 있는 `langchain-community` 패키지 자체도 sunset(단계적 종료)이
> 공식 발표되었습니다. 다만 현재 시점에는 `KuzuGraph` / `KuzuQAChain` 을 대체할
> 독립 패키지가 따로 나오지 않아서, 이 노트북은 여전히
> `langchain_community.graphs.kuzu_graph.KuzuGraph` 와
> `langchain_community.chains.graph_qa.kuzu.KuzuQAChain` 을 사용합니다.
> 실행하면 `DeprecationWarning` 이 뜨는 게 정상이며, 학습 목적의 실습에서는
> 무시하고 진행해도 무방합니다. (실제로 동작하는지 직접 검증 후 작성되었습니다.)

In [ ]:
ensure("langchain")
ensure("langchain_openai", "langchain-openai")
ensure("langchain_community", "langchain-community")
ensure("dotenv", "python-dotenv")

import kuzu
print("kuzu version:", kuzu.__version__)


---
## STEP 2 · [기본 실습] 스키마 정의 → 노드/관계 생성 → Cypher 조회

먼저 **서버 없이 KùzuDB가 어떻게 동작하는지** 직접 손으로 익혀봅니다.
주제는 "병해충(Pest) - 작물(Crop)" 관계입니다.

### 2-1. 로컬 DB 생성

`kuzu.Database("./kuzu_db")` 를 실행하면 `./kuzu_db` 라는 **폴더**가 생성되고,
이 폴더 자체가 데이터베이스입니다. 클라우드 접속, 계정, 인증 같은 절차가 전혀 없습니다.

In [ ]:
import kuzu

db = kuzu.Database("./kuzu_db")
conn = kuzu.Connection(db)

print("✅ DB 연결 완료 — ./kuzu_db 폴더가 곧 데이터베이스입니다.")


### 2-2. 스키마 생성

그래프를 만들기 전에, 어떤 종류의 노드와 관계가 있을지 먼저 정의합니다.
- `Pest` 노드: 병해충 (이름이 기본키)
- `Crop` 노드: 작물 (이름이 기본키)
- `Infects` 관계: Pest → Crop, 피해 정도(severity) 속성 포함

In [ ]:
conn.execute("CREATE NODE TABLE IF NOT EXISTS Pest(name STRING, PRIMARY KEY(name))")
conn.execute("CREATE NODE TABLE IF NOT EXISTS Crop(name STRING, PRIMARY KEY(name))")
conn.execute("CREATE REL TABLE IF NOT EXISTS Infects(FROM Pest TO Crop, severity STRING)")

print("✅ 스키마 생성 완료: Pest, Crop, Infects")


### 2-3. 노드와 관계 생성 (Cypher로 직접 입력)

병해충 3종, 작물 3종을 만들고 각각 INFECTS 관계로 연결합니다.

In [ ]:
# 병해충 노드 생성
conn.execute("CREATE (:Pest {name: '진딧물'})")
conn.execute("CREATE (:Pest {name: '응애'})")
conn.execute("CREATE (:Pest {name: '노균병'})")

# 작물 노드 생성
conn.execute("CREATE (:Crop {name: '사과나무'})")
conn.execute("CREATE (:Crop {name: '배나무'})")
conn.execute("CREATE (:Crop {name: '장미'})")

# 관계 생성 (MATCH로 두 노드를 찾은 뒤 CREATE로 연결)
conn.execute("""
    MATCH (p:Pest {name: '진딧물'}), (c:Crop {name: '사과나무'})
    CREATE (p)-[:Infects {severity: '보통'}]->(c)
""")
conn.execute("""
    MATCH (p:Pest {name: '진딧물'}), (c:Crop {name: '장미'})
    CREATE (p)-[:Infects {severity: '심함'}]->(c)
""")
conn.execute("""
    MATCH (p:Pest {name: '응애'}), (c:Crop {name: '배나무'})
    CREATE (p)-[:Infects {severity: '심함'}]->(c)
""")
conn.execute("""
    MATCH (p:Pest {name: '노균병'}), (c:Crop {name: '배나무'})
    CREATE (p)-[:Infects {severity: '낮음'}]->(c)
""")

print("✅ 노드 6개, 관계 4개 생성 완료")


### 2-4. Cypher로 조회하기

이제 만든 그래프를 질문 형태로 조회해봅니다.

In [ ]:
# 조회 1: 진딧물이 영향을 주는 작물 전체
result = conn.execute("""
    MATCH (p:Pest {name: '진딧물'})-[r:Infects]->(c:Crop)
    RETURN c.name AS 작물, r.severity AS 피해정도
""")
print("🔎 진딧물이 영향을 주는 작물:")
print(result.get_as_df())


In [ ]:
# 조회 2: 배나무에 영향을 주는 병해충 전체
result = conn.execute("""
    MATCH (p:Pest)-[r:Infects]->(c:Crop {name: '배나무'})
    RETURN p.name AS 병해충, r.severity AS 피해정도
""")
print("🔎 배나무에 영향을 주는 병해충:")
print(result.get_as_df())


In [ ]:
# 조회 3: 피해 정도가 '심함'인 관계만 필터링
result = conn.execute("""
    MATCH (p:Pest)-[r:Infects {severity: '심함'}]->(c:Crop)
    RETURN p.name AS 병해충, c.name AS 작물
""")
print("🔎 심각한 피해를 주는 병해충-작물 조합:")
print(result.get_as_df())


> 💡 `./kuzu_db` 폴더를 통째로 복사하거나 압축하면 그래프 DB 전체를 옮길 수 있습니다.
> 다른 컴퓨터에서 작업을 이어가려면 이 폴더만 그대로 가져가면 됩니다.

---
## STEP 3 · [CSV 실습] 대량 데이터 적재하기

손으로 몇 개씩 만드는 건 실습용이고, 실무에서는 CSV로 대량 데이터를 한 번에 넣습니다.
이번엔 별도의 새 DB(`./kuzu_db_csv`)에, 미리 준비된 CSV 3개를 적재합니다.

- `data/pest.csv`: 병해충 8종 (name, category)
- `data/crop.csv`: 작물 6종 (name, type)
- `data/infects.csv`: 병해충-작물 관계 12건 (pest_name, crop_name, severity)

In [ ]:
import pandas as pd

print("=== pest.csv ===")
print(pd.read_csv("data/pest.csv"))
print("\n=== crop.csv ===")
print(pd.read_csv("data/crop.csv"))
print("\n=== infects.csv ===")
print(pd.read_csv("data/infects.csv"))


### 3-1. 새 DB 생성 및 스키마 정의

이번엔 `category`, `type` 속성이 추가된 스키마를 만듭니다.

In [ ]:
db_csv = kuzu.Database("./kuzu_db_csv")
conn_csv = kuzu.Connection(db_csv)

conn_csv.execute("CREATE NODE TABLE IF NOT EXISTS Pest(name STRING, category STRING, PRIMARY KEY(name))")
conn_csv.execute("CREATE NODE TABLE IF NOT EXISTS Crop(name STRING, type STRING, PRIMARY KEY(name))")
conn_csv.execute("CREATE REL TABLE IF NOT EXISTS Infects(FROM Pest TO Crop, severity STRING)")

print("✅ kuzu_db_csv 스키마 생성 완료")


### 3-2. COPY FROM 으로 CSV 적재

`COPY ... FROM "파일경로" (header=true)` 한 줄로 CSV 전체를 그래프에 넣습니다.

In [ ]:
conn_csv.execute('COPY Pest FROM "data/pest.csv" (header=true)')
conn_csv.execute('COPY Crop FROM "data/crop.csv" (header=true)')
conn_csv.execute('COPY Infects FROM "data/infects.csv" (header=true)')

print("✅ CSV 3개 적재 완료")


### 3-3. 적재 확인

In [ ]:
pest_count = conn_csv.execute("MATCH (p:Pest) RETURN count(p) AS cnt").get_as_df()
crop_count = conn_csv.execute("MATCH (c:Crop) RETURN count(c) AS cnt").get_as_df()
rel_count = conn_csv.execute("MATCH ()-[r:Infects]->() RETURN count(r) AS cnt").get_as_df()

print("Pest 노드 수:", pest_count['cnt'][0])
print("Crop 노드 수:", crop_count['cnt'][0])
print("Infects 관계 수:", rel_count['cnt'][0])


In [ ]:
# category가 '해충'인 Pest 중 severity가 '심함'인 관계만 조회
result = conn_csv.execute("""
    MATCH (p:Pest)-[r:Infects {severity: '심함'}]->(c:Crop)
    WHERE p.category = '해충'
    RETURN p.name AS 병해충, c.name AS 작물, r.severity AS 피해정도
""")
df = result.get_as_df()
print("🔎 '해충' 카테고리 중 심각한 피해를 주는 사례:")
df


---
## STEP 4 · [Graph RAG 실습] LangChain 연동 — 자연어로 그래프 질문하기

여기서부터가 진짜 **Graph RAG**입니다. 자연어 질문을 LLM이 Cypher로 변환하고,
그 결과를 다시 자연어 답변으로 만들어줍니다.

### 4-1. API 키 로드

`.env` 파일에서 `OPENAI_API_KEY`를 읽어옵니다. 이 노트북이 `day3/02_graph_rag/` 에 있다면
`.env`는 보통 프로젝트 루트(`../../.env`)에 있습니다. 경로가 다르면 아래 `dotenv_path`를 수정하세요.

In [ ]:
from dotenv import load_dotenv
import os

dotenv_path = "../../.env"
load_dotenv(dotenv_path)

if os.getenv("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY 로드 완료")
else:
    print("⚠️ OPENAI_API_KEY를 찾을 수 없습니다. dotenv_path를 확인하세요:", dotenv_path)


### 4-2. KuzuGraph로 그래프 연결

기존에 만든 `./kuzu_db_csv` 를 LangChain의 `KuzuGraph` 로 감싸서,
LLM이 그래프 스키마를 읽고 Cypher를 생성할 수 있게 합니다.

In [ ]:
from langchain_community.graphs import KuzuGraph

graph = KuzuGraph(db_csv, allow_dangerous_requests=True)
graph.refresh_schema()

print("📋 그래프 스키마:")
print(graph.get_schema)


### 4-3. KuzuQAChain 생성

`KuzuQAChain` 은 내부적으로 4단계를 거칩니다:

```
질문(자연어) → ① Cypher 쿼리 생성(LLM) → ② 그래프에서 실행 → ③ 결과 취합 → ④ 자연어 답변 생성(LLM)
```

In [ ]:
from langchain_community.chains.graph_qa.kuzu import KuzuQAChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = KuzuQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
)

print("✅ Graph RAG 체인 준비 완료")


### 4-4. 질문해보기

아래 질문들을 하나씩 실행해보면서, 생성된 Cypher와 답변을 함께 확인합니다.

In [ ]:
def ask(question: str):
    try:
        result = chain.invoke(question)
        print("💬 답변:", result.get("result", result))
    except Exception as e:
        print("⚠️ 스키마에서 답을 찾을 수 없습니다. 다른 질문을 시도해보세요.")
        print("(에러 내용:", e, ")")

ask("진딧물이 영향을 주는 작물은 뭐야?")


In [ ]:
ask("심함 단계로 피해를 주는 병해충은 뭐가 있어?")


In [ ]:
ask("사과나무에 영향을 주는 해충 종류는?")


In [ ]:
# 의도적으로 스키마 밖의 질문 — 그래프에 없는 걸 지어내지 않는지 확인
ask("오늘 날씨 어때?")


### 4-5. 직접 질문해보기

아래 셀의 `question` 값을 바꿔서 자유롭게 질문해보세요.

In [ ]:
question = "배추에 영향을 주는 병해충의 피해 정도를 알려줘"
ask(question)


---
## 🚀 추가 실습 — 기능 확장

아래는 직접 셀을 추가해서 시도해볼 수 있는 확장 과제입니다.

**과제 1 · 응답 시간 측정**
`time` 모듈로 `chain.invoke()` 호출 전후 시간을 측정해서, 질문마다 응답 시간을 비교해보세요.

**과제 2 · 더 큰 데이터셋**
`data/pest.csv`, `crop.csv`, `infects.csv` 를 각각 더 많은 행으로 늘려서 다시 적재하고,
적재 시간과 조회 속도가 어떻게 변하는지 비교해보세요.

**과제 3 · 벡터 검색 결합**
병해충 설명(description) 컬럼을 추가하고 임베딩한 뒤, KùzuDB의 벡터 인덱스 기능으로
"증상이 비슷한 병해충 찾기" 기능을 만들어보세요.

**과제 4 · Streamlit 웹앱으로 옮기기**
이 노트북의 STEP 4 코드를 `streamlit_app.py` 로 옮겨서, 질문 입력창 + 답변 표시 +
"생성된 Cypher 보기" expander로 구성된 웹앱을 만들어보세요.

---

## ❓ 자주 묻는 질문

**Q. `langchain-kuzu` 패키지를 따로 설치해야 하나요? 경고 메시지가 뜨는데 괜찮나요?**
독립 패키지였던 langchain-kuzu는 archived 상태이며, 이를 포함하던 langchain-community 패키지 자체도
sunset(단계적 종료)이 공식 발표되었습니다. 그래서 import 시 `DeprecationWarning` 이 뜨는 게 정상입니다.
다만 현재 KuzuGraph/KuzuQAChain을 대체하는 독립 패키지가 아직 없어서, 코드는 그대로 정상 동작합니다
(이 노트북도 실제 실행 검증을 거쳤습니다). 추후 공식 마이그레이션 안내가 나오면 그때 코드를 옮기면 됩니다.

**Q. `kuzu_db`, `kuzu_db_csv` 폴더를 지우고 다시 만들면 안 되나요?**
가능합니다. KùzuDB는 폴더 하나가 곧 데이터베이스이므로, 폴더를 삭제(`rm -rf kuzu_db`)하고
이 노트북을 처음부터 다시 실행하면 깨끗한 상태에서 새로 시작됩니다.

**Q. KuzuQAChain이 이상한 Cypher를 생성해요**
질문에 등장하는 단어가 그래프 안의 실제 값(예: '진딧물', '심함')과 정확히 일치하지 않으면
빈 결과나 잘못된 Cypher가 나올 수 있습니다. `verbose=True` 로 설정해두면 생성된 Cypher를
바로 확인할 수 있어 디버깅이 쉽습니다.

**Q. 노트북을 다시 실행(Restart & Run All)하면 에러가 나요**
`CREATE NODE TABLE` 부분에 `IF NOT EXISTS` 를 넣어두었지만, `CREATE (:Pest {...})` 같은
데이터 생성 코드는 다시 실행하면 같은 노드가 중복 생성되거나 기본키 충돌 에러가 날 수 있습니다.
처음부터 다시 실행하려면 `kuzu_db`, `kuzu_db_csv` 폴더를 먼저 삭제하세요.
